# Appendix A: PyTorch 简介
A.1 什么是PyTorch
PyTorch是一个开源的基于Python的深度学习库。
## A.1 Pytorch的三大核心组件
PyTorch有三大核心组件：张量库、自动微分引擎和深度学习库。
- 张量库：用于高效计算的张量（数组）库
- 自动微分引擎：用于计算自动微分的实用工具
- 深度学习库：提供了模块化、灵活且高效的构建块
### A.1.2 定义深度学习
- 人工智能：模仿人类智能
- 机器学习：从数据中学习，是人工智能的子类别
- 深度学习：使用深度神经网络的机器学习，是机器学习的子类别

In [ ]:
# 检查 PyTorch 的版本
import torch
torch.__version__

In [ ]:
# 检查是否支持 Nvidia GPU
import torch
torch.cuda.is_available()

In [ ]:
# 检查是否支持 Apple Silicon 加速 PyTorch
print(torch.backends.mps.is_available())

## A.2 理解张量
张量是可以通过阶数表征的数学对象，阶数提供了张量的维度。
张量的秩是张量的维度。标量的秩为0，向量的秩为1，矩阵的秩为2。

PyTorch 张量类似于 NumPy 数组，但在 NumPy 数组基础上还具有自动微分（简化梯度计算）、支持 GPU 计算（加速神经网络训练）等特性，
但是数组操作的语法和 NumPy 类似。
### A.2.1 标量、向量、矩阵和张量

In [ ]:
# 创建 PyTorch 张量
import torch

tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2],
                         [3, 4]])
tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]])

print(tensor0d)
print(tensor1d)
print(tensor2d)
print(tensor3d)

### A.2.2 张量的数据类型
PyTorch 张量默认采用64位整数类型，可以使用.dtype访问张量的数据类型。

In [ ]:
# 查看张量数据类型
tensor1d = torch.tensor([1, 2, 3])
print(tensor1d.dtype)

In [ ]:
floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

In [ ]:
# 使用.to方法更改精度
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

### A.2.3 常见的 PyTorch 张量操作

In [ ]:
# 创建张量
tensor2d = torch.tensor([[1, 2, 3],
                         [4, 5, 6]])
# 查看张量的形状
print(tensor2d.shape)

In [ ]:
# 更改张量的形状
print(tensor2d.reshape(3, 2))

In [ ]:
# 重塑张量
print(tensor2d.view(3, 2))

reshape()和view()的区别在于：view()要求原始数据是连续的，而reshape()不要求。

In [ ]:
# 转置张量
print(tensor2d)
print(tensor2d.T)

In [ ]:
# 矩阵乘法
print(tensor2d.matmul(tensor2d.T))

In [ ]:
# 矩阵乘法@
print(tensor2d @ tensor2d.T)

## A.3 将模型视为计算图
PyTorch 的自动微分引擎是 autograd，其能够实现在动态计算图中的梯度自动计算。

In [ ]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

In [ ]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

loss.backward()
print(w1.grad)
print(b.grad)

## A.5 实现多层神经网络
在 PyTorch 中实现神经网络可以通过子类 torch.nn.Module 自定义网络架构。

In [ ]:
import torch

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 第一个隐藏层
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 第二个隐藏层
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # 输出层
            torch.nn.Linear(20, num_outputs)
        )
    
    def forward(self, x):
        logits = self.layers(x)
        return logits

In [ ]:
model = NeuralNetwork(50, 3)
# 查看模型结构摘要
print(model)

In [ ]:
# 检查模型可训练参数总数
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters: ", num_params)

In [ ]:
# 访问神经网络中一层参数的权重
print(model.layers[0].weight)
# 查看该层参数的维度
print(model.layers[0].weight.shape)

In [ ]:
# 使用随机数初始化种子使得随机权重可复现
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

In [ ]:
# 通过前向传播使用模型
torch.manual_seed(123)
X = torch.rand((1, 50))
out = model(X)
print(out)

In [ ]:
# 如果只是使用模型进行推理，而无需训练（即反向传播），使用 torch.no_grad() 不跟踪梯度信息
with torch.no_grad():
    out = model(X)
print(out)

In [ ]:
# 使用 softmax 计算类别成员概率
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

## A.6 设置高效的数据加载器
数据加载器在模型训练过程中迭代使用，PyTorch 中数据加载的整体思路是：

自定义的 Dataset 类 --> 训练数据集 --> 训练数据加载器

自定义的 Dataset 类 --> 测试数据集 --> 测试数据加载器

DataLoader 类 --> 训练数据加载器

DataLoader 类 --> 测试数据加载器

In [ ]:
# 创建一个简单的示例数据集
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5],
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])
y_test = torch.tensor([0, 1])

In [ ]:
# 创建自定义数据集类 ToyDataset
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        """设置可以在 __getitem__ 和 __len__ 方法中可以访问的属性"""
        self.features = X
        self.labels = y
    
    def __getitem__(self, index):
        """定义了通过索引返回数据集中单个项目的具体指令"""
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    
    def __len__(self):
        """检索数据集长度的指令"""
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [ ]:
# 实例化数据加载器
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True, # 是否打乱数据
    num_workers=0, # 用于并行加载和预处理数据 =0 表示数据加载在主进程上单独进行，>0 会启动多个工作进程并行加载数据，从而释放主进程专注于训练模型
    drop_last=True # 丢弃最后一个批次，如果最后一个批次的数据量显著小于其他批次
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [ ]:
# 一个训练轮次，每个训练示例刚好访问一次，每次被访问的这一组数据称为一个批次（batch）
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

# A.7 典型的训练循环

In [ ]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.5
)

num_epochs = 3
for epoch in range(num_epochs):
    # 将模型设置为训练模式
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)

        loss = F.cross_entropy(logits, labels)
        # 将上一轮的梯度设置为0，以防止意外的梯度累积
        optimizer.zero_grad()
        loss.backward()
        # 优化器使用梯度更新模型参数
        optimizer.step()

        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")
    
# 将模型设置为评估模式
model.eval()
with torch.no_grad():
    outputs = model(X_train)
# 为了获得类别成员概率，使用 PyTorch 的 softmax 函数
torch.set_printoptions(sci_mode=False) # 让输出更加易读
probas = torch.softmax(outputs, dim=1)
print(probas)

In [ ]:
# 使用 PyTorch 的 argmax 函数将概率值转换为类别标签预测
predictions = torch.argmax(probas, dim=1)
print(predictions)

In [ ]:
# 比较预测类别和真实类别
predictions == y_train

In [ ]:
# 使用 torch.sum() 计算正确预测的数量
torch.sum(predictions == y_train)

In [ ]:
# 计算预测准确率
def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)
        
        predictions = torch.argmax(logits, dim=1)
        compare = (predictions == labels)
        correct += torch.sum(compare)
        total_examples += len(compare)
    
    return (correct / total_examples).item() # item() 将张量的值以 Python 浮点数的形式返回

In [ ]:
# 将 compute_accuracy 函数应用于训练数据
print(compute_accuracy(model, train_loader))
print(compute_accuracy(model, test_loader))

## A.8 保存和加载模型
在模型训练完成后保存模型，以便后续重用。在 PyTorch 中保存和加载模型使用 torch.save()。

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
# 加载模型
model = NeuralNetwork(2, 2) # 应用模型实例
model.load_state_dict(torch.load("model.pth")) # 加载参数

## A.9 使用 GPU 优化训练性能
使用 GPU 加速深度神经网络训练，相比于 CPU，GPU 能够显著提升模型训练速度。
### A.9.1 在 GPU 设备上运行 PyTorch
在 PyTorch 中，设备是执行计算和存储数据的地方，CPU 和 GPU 都是设备的示例。如果一个张量存放在某个设备上，那么其操作也在该设备上进行。

In [ ]:
# 因为我使用的是 Macbook Air M2，没有CUDA，因此使用 Apple Silicon 的 MPS
print(torch.backends.mps.is_available())

In [ ]:
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])
print(tensor_1 + tensor_2)

In [ ]:
# 使用 .to() 将这些张量转移到 GPU 上
tensor_1 = tensor_1.to("mps")
tensor_2 = tensor_2.to("mps")
print(tensor_1 + tensor_2)

输出中的 device 表明当前张量在第一个 GPU 上，即 mps:0。如果有多个 GPU，可以指定将张量转移到哪一个 GPU 上，通过在传输命令中指定设备 ID 实现，比如 .to("cuda:1")、.to("mps:1") 等。

然而，参与计算的所有张量必须位于同一个设备上，例如一个张量位于 CPU，另一个位于 GPU，计算时就会失败。

In [ ]:
tensor_1 = tensor_1.to("cpu")
tensor_2 = tensor_2.to("mps")
print(tensor_1 + tensor_2)

### A.9.2 单个 GPU 训练
修改训练循环，使模型训练在 GPU 上进行。

In [50]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


In [52]:
# 练习 A.4
import torch

# CPU
a = torch.rand(100, 200)
b = torch.rand(200, 300)
%timeit a @ b

# GPU
device = "mps" if torch.backends.mps.is_available() else "cpu"
a, b = a.to(device), b.to(device)
%timeit a @ b

15.9 μs ± 176 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
27.9 μs ± 220 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### A.9.3 使用多个 GPU 训练
分布式训练是将模型训练分配到多个 GPU 和机器上，通过这种方法可以可以显著减少训练时间。由于本机是单 GPU 环境，这部分内容暂不考虑。